# Select grid cells with one GFED class per TEM community

Run all cells in order in a fresh kernel. Change paths and thresholds only in Step 1 for a normal run.

The workflow first applies the original TEM community-cover screen. For communities with multiple compatible GFED classes, each complete positive-fire grid cell can cast one vote for its dominant compatible burned-area class. Community-level support then determines at most one GFED class for that TEM community across all locations. Single-class mappings are retained directly. Disabled or unresolved communities receive no automatic selection.

Voting measures burned-area contribution, not vegetation cover. The vote denominator uses only compatible GFED classes. It does not measure their contribution to all fire in the cell. A class is never selected by summing burned area across different grid cells or by choosing the lowest class ID in a tie.

After the class is fixed, every LULCC-qualified cell for that community is assessed using that class only, including cells that abstained or voted for another class. The original fire filters, percentile bins, target quantiles, shortlist, ranking, fallback, no-reuse rule, and manual replacements are retained. No secondary class is tried if the chosen class fails the later filters.

The original notebooks are unchanged. This notebook writes to a separate output folder.


## 1. Settings and paths


In [ ]:
import json
import sqlite3
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


# Configuration: edit paths and settings in this section.

# Output directory used by Stages 1 and 2. Update this path for your server.
OUTPUT_DIR = Path('/quobyte/emoniergrp/luyy123/calibration_gridcell_selection_2002_2022_v2')

# Use the same inclusive year range as Stages 1 and 2: 2002-2022 = 21 years.
START_YEAR = 2002
END_YEAR = 2022
N_YEARS = END_YEAR - START_YEAR + 1

# Read the full_1_23 Stage 1 files, rather than the earlier communities 1-14 files.
FRAC_FILE = OUTPUT_DIR / f'01_community_fraction_{START_YEAR}_{END_YEAR}_full_1_23.csv'
STAGE1_RUN_SUMMARY = OUTPUT_DIR / f'01_run_summary_{START_YEAR}_{END_YEAR}_full_1_23.json'

# Stage 2 outputs must cover every grid cell required by the current Stage 1 data.
GFED_MEAN = OUTPUT_DIR / f'02_gfed_by_class_summary_{START_YEAR}_{END_YEAR}.csv'
GFED_ANNUAL = OUTPUT_DIR / f'02_gfed_annual_nonzero_{START_YEAR}_{END_YEAR}.csv'

# Write Stage 3 outputs to a separate subdirectory.
# Change RESULT_DIR before each run if you want to keep multiple trials.
# Reusing RESULT_DIR replaces result tables but preserves manual overrides.
RESULT_DIR = OUTPUT_DIR / f'stage3_full_1_23_{START_YEAR}_{END_YEAR}_unique_gfed_voting'

# Location for a temporary SQLite database; use scratch storage if needed.
# Each run creates and removes its own temporary directory.
WORK_DIR = OUTPUT_DIR

# Number of CSV rows read at a time; this does not change the calculations.
# Eligible candidates and matched GFED summary records are still retained in memory.
CSV_CHUNKSIZE = 150_000

# Original community dominance and stability thresholds.
MIN_MEAN_COMMUNITY_FRACTION = 0.60
MIN_MIN_COMMUNITY_FRACTION = 0.40
MIN_FIRE_YEARS = 3

# Select low, medium, then high fire, without reusing a cell within a community.
# Targets are the 30th/50th/70th percentiles; bin boundaries remain 40% and 60%.
TARGET_QUANTILES = {'low_fire': 0.30, 'medium_fire': 0.50, 'high_fire': 0.70}
SHORTLIST_K = 20

# New voting thresholds. These are adjustable working choices, not validated optima.
# Apply these four thresholds only to communities with multiple compatible classes.

# A grid votes only when one class supplies at least this fraction of the
# 21-year burned area summed over its compatible GFED classes. Exact ties abstain.
MIN_GRID_VOTE_SHARE = 0.60

# The winning class must receive at least this fraction of all clear grid votes.
# This is a fraction of votes, not burned area. Exact community ties are unresolved.
MIN_COMMUNITY_WINNER_SHARE = 0.60

# Clear-voting grids must represent at least this fraction of all grid cells
# passing the LULCC screen for this community. Zero-fire and incomplete grids
# remain in this denominator but cannot cast votes.
MIN_COMMUNITY_VOTER_FRACTION = 0.50

# Require this many clear-voting grids in addition to the two community fractions.
MIN_COMMUNITY_VOTES = 3

# By default, require a complete Stage 1 run with no unclassified area.
# Set True only if Stage 1 allows unclassified area in the total-area denominator.
ALLOW_STAGE1_UNCLASSIFIED_AREA = False

# Flag annual burned area greater than the whole grid-cell area for review.
# False: flag only; do not remove candidates, cap values, or change selection.
# True: exclude flagged candidates before recalculating percentiles and selections.
# A flag alone does not establish a product error; repeated burning needs review.
EXCLUDE_QC_FLAGGED_CANDIDATES = False
ANNUAL_AREA_REVIEW_RATIO = 1.0


## 2. Compatible classes

The original 23 communities and compatibility lists are retained. Multiple IDs are alternatives for voting, not burned areas to combine for the final fire metric. Empty lists remain disabled. Existing proxy limitations still apply.


In [ ]:
# Preserve the community IDs supplied in the original script.
COMMUNITY_NAMES = {
    1: 'Ice', 2: 'Alpine Tundra and Polar Desert', 3: 'Moist and Wet Tundra',
    4: 'Boreal Forests', 5: 'Temperate Coniferous Forests',
    6: 'Temperate Deciduous Forests', 7: 'Temperate Broadleaved Evergreen Forest',
    8: 'Short Grassland', 9: 'Tall Grassland', 10: 'Mediterranean Shrubland',
    11: 'Xeric Shrubland', 12: 'Xeric Woodland',
    13: 'Tropical Evergreen Forests', 14: 'Tropical Deciduous Forests',
    15: 'Impervious Surfaces', 16: 'Turf Lawn', 17: 'Soybean', 18: 'Maize',
    19: 'Fruit tree', 20: 'Switchgrass', 21: 'Vegetable Farm',
    22: 'Pasture', 23: 'Rangeland',
}

GFED_NAMES = {
    1: 'Tundra', 2: 'Sparse boreal forest', 3: 'Boreal forest',
    4: 'Temperate grassland', 5: 'Temperate shrubland', 6: 'Temperate mosaic',
    7: 'Temperate forest', 8: 'Tropical grassland', 9: 'Tropical shrubland',
    10: 'Open savanna', 11: 'Woody savanna', 12: 'Tropical forest',
    13: 'Other', 14: 'Cropland', 15: 'Peat', 16: 'Deforestation',
}

# Register all 23 communities; an empty tuple disables GFED-based selection.
# Communities 2-14 retain the proposed compatibility mapping, not an official crosswalk.
# Communities 16 and 22 retain GFED 4 and 8 as grassland proxy alternatives.
# Communities 17-21 use generic Cropland; crop-specific fire cannot be separated.
# Communities 19/20 assume managed orchards/energy crops; review if definitions change.
# Communities 1 and 15 have no direct match; do not assign them to Other.
# Community 23 remains deferred, as configured in the original script.
COMMUNITY_TO_GFED = {
    1: (),
    2: (1,), 3: (1,), 4: (2, 3), 5: (7,), 6: (7,), 7: (7,),
    8: (4, 8), 9: (4, 8), 10: (5,), 11: (5, 9),
    12: (6, 10, 11), 13: (12,), 14: (12,),
    15: (),
    16: (4, 8),
    17: (14,), 18: (14,), 19: (14,), 20: (14,), 21: (14,),
    22: (4, 8),
    23: (),
}

# Preserve the original approximation flags and flag every added proxy.
# False does not imply exact equivalence; communities 2-14 remain proposed matches.
APPROXIMATE_CROSSWALK = {4, 8, 9, 11, 12, 14, 16, 17, 18, 19, 20, 21, 22}

# Shared keys and required Stage 1 columns.

KEY = ['lon', 'lat', 'community_type']
FRACTION_COLUMNS = [
    'lon', 'lat', 'community_type', 'community_name',
    'mean_community_fraction', 'min_community_fraction', 'max_community_fraction',
    'n_years_grid',
]

rows = []
for comm in range(1, 24):
    ids = COMMUNITY_TO_GFED[comm]
    if comm in (1, 15):
        (status, note) = ('no_direct_match', 'No direct GFED class is assigned; selection is disabled.')
    elif comm == 23:
        (status, note) = ('deferred_by_user', 'Selection remains deferred as specified in the source configuration.')
    elif comm in (16, 22):
        (status, note) = ('user_confirmed_grassland_proxy', 'GFED classes 4 and 8 are alternative proxies; turf or pasture fire cannot be isolated.')
    elif 17 <= comm <= 21:
        (status, note) = ('proposed_generic_cropland_proxy', 'GFED Cropland is a generic proxy and cannot isolate individual crops.')
    else:
        (status, note) = ('original_proposed_compatibility', 'Proposed compatibility retained from the source; not an official one-to-one crosswalk.')
    approximate = None
    if ids:
        approximate = comm in APPROXIMATE_CROSSWALK
    id_text = []
    name_text = []
    for gfed_id in ids:
        id_text.append(str(gfed_id))
        name_text.append(GFED_NAMES[gfed_id])
    rows.append({
        'community_type': comm, 'community_name': COMMUNITY_NAMES[comm],
        'selection_enabled': bool(ids),
        'gfed_class_ids': ','.join(id_text),
        'gfed_class_names': '; '.join(name_text),
        'mapping_approximate': approximate,
        'mapping_status': status, 'mapping_note': note,
    })
crosswalk = pd.DataFrame(rows)
crosswalk = crosswalk.rename(columns={
    'gfed_class_ids': 'compatible_gfed_class_ids',
    'gfed_class_names': 'compatible_gfed_class_names',
})
display(crosswalk)


## 3. Prepare inputs and output paths

Require a completed Stage 1 run with matching years. Missing required class-summary rows remain errors; they are not zero-fire observations. Vote and mapping tables record the new decisions without adding repeated run-status logs.


In [ ]:
if START_YEAR > END_YEAR or CSV_CHUNKSIZE < 1 or SHORTLIST_K < 1:
    raise ValueError('Use a valid year range, positive chunk size, and positive shortlist size.')
if not 1 <= MIN_FIRE_YEARS <= N_YEARS:
    raise ValueError('MIN_FIRE_YEARS must be between 1 and N_YEARS.')

if not 0 < MIN_GRID_VOTE_SHARE <= 1:
    raise ValueError('MIN_GRID_VOTE_SHARE must be greater than 0 and at most 1.')
if not 0 < MIN_COMMUNITY_WINNER_SHARE <= 1:
    raise ValueError('MIN_COMMUNITY_WINNER_SHARE must be greater than 0 and at most 1.')
if not 0 <= MIN_COMMUNITY_VOTER_FRACTION <= 1:
    raise ValueError('MIN_COMMUNITY_VOTER_FRACTION must be between 0 and 1.')
if MIN_COMMUNITY_VOTES < 1 or MIN_COMMUNITY_VOTES % 1 != 0:
    raise ValueError('MIN_COMMUNITY_VOTES must be a positive integer.')

stage1 = json.loads(STAGE1_RUN_SUMMARY.read_text(encoding='utf-8'))
allowed_status = ['complete']
if ALLOW_STAGE1_UNCLASSIFIED_AREA:
    allowed_status.append('complete_with_unclassified_area')
if stage1.get('status') not in allowed_status:
    raise ValueError('Stage 1 is incomplete or has unapproved unclassified area.')
if (stage1['source']['start_year'], stage1['source']['end_year']) != (START_YEAR, END_YEAR):
    raise ValueError('Stage 1 years do not match the configured years.')

RESULT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
paths = {
    'crosswalk': RESULT_DIR / '03_community_gfed_crosswalk.csv',
    'grid_votes': RESULT_DIR / '03_gfed_grid_votes.csv',
    'voting': RESULT_DIR / '03_community_gfed_voting.csv',
    'candidates': RESULT_DIR / '03_all_eligible_candidates.csv',
    'shortlist': RESULT_DIR / '03_ranked_shortlists.csv',
    'selected': RESULT_DIR / '03_selected_observation_only.csv',
    'summary': RESULT_DIR / '03_community_screening_summary.csv',
    'overrides': RESULT_DIR / '03_manual_overrides.csv',
}
print('Output directory:', RESULT_DIR)


## 4. Screen community dominance and stability

Read the Stage 1 fraction table in chunks. A candidate must have all configured years, a mean community fraction of at least 0.60, and a minimum annual community fraction of at least 0.40. Only communities with an assigned GFED mapping enter the candidate table.

Chunked reading limits input-buffer size. Candidate records are still stored in memory. `total` and `passed` retain counts for all communities, including those without a GFED mapping.


In [ ]:
total = {}
passed = {}
for comm in COMMUNITY_NAMES:
    total[comm] = 0
    passed[comm] = 0
parts = []
enabled = []
for comm, ids in COMMUNITY_TO_GFED.items():
    if len(ids) > 0:
        enabled.append(comm)
for chunk in pd.read_csv(FRAC_FILE, chunksize=CSV_CHUNKSIZE):
    if not chunk.community_type.isin(COMMUNITY_NAMES).all():
        raise ValueError('Stage 1 contains missing community IDs or IDs outside 1-23.')
    chunk['lon'] = chunk.lon.round(6)
    chunk['lat'] = chunk.lat.round(6)
    # Use the supplied names; Stage 1 has already classified currentveg/agstate.
    chunk['community_name'] = chunk.community_type.map(COMMUNITY_NAMES)
    ok = ((chunk.n_years_grid == N_YEARS) &
          (chunk.mean_community_fraction >= MIN_MEAN_COMMUNITY_FRACTION) &
          (chunk.min_community_fraction >= MIN_MIN_COMMUNITY_FRACTION))
    for c, n in chunk.community_type.value_counts().items():
        total[int(c)] += int(n)
    for c, n in chunk.loc[ok, 'community_type'].value_counts().items():
        passed[int(c)] += int(n)
    use = ok & chunk.community_type.isin(enabled)
    if use.any():
        parts.append(chunk.loc[use].copy())
if parts:
    eligible = pd.concat(parts, ignore_index=True)
else:
    eligible = pd.DataFrame(columns=FRACTION_COLUMNS)
if eligible.duplicated(KEY).any():
    raise ValueError('Stage 1 contains duplicate grid-cell/community rows; check the input.')
print(f'LULCC records: {sum(total.values()):,}; mapped candidates passing the fraction screen: {len(eligible):,}', flush=True)
display(eligible.head())


## 5. Read separate GFED summary records

Read every compatible GFED class for each LULCC-qualified grid. Retain separate class totals. Reject missing or duplicate summary records and invalid input values. A grid can vote only if every compatible class has all configured years; incomplete existing records cause that grid to abstain.

CSV input is read in chunks. Matched candidate records are retained in memory, as in the source notebook.


In [ ]:
if not eligible.empty:
    parts = []
    for comm, ids in COMMUNITY_TO_GFED.items():
        src = eligible.loc[eligible.community_type == comm, KEY]
        for gfed in ids:
            x = src.copy()
            x['gfed_id'] = gfed
            parts.append(x)
    links = pd.concat(parts, ignore_index=True)

    parts = []
    need = ['lon', 'lat', 'gfed_id', 'gfed_valid_years', 'gfed_ba_sum_km2', 'grid_area_km2']
    for chunk in pd.read_csv(GFED_MEAN, usecols=need, chunksize=CSV_CHUNKSIZE):
        chunk['lon'] = chunk.lon.round(6)
        chunk['lat'] = chunk.lat.round(6)
        m = chunk.merge(links, on=['lon', 'lat', 'gfed_id'], how='inner')
        if not m.empty:
            parts.append(m)
    if parts:
        matched = pd.concat(parts, ignore_index=True)
    else:
        matched = pd.DataFrame(columns=need + ['community_type'])
    if matched.duplicated(KEY + ['gfed_id']).any():
        raise ValueError('Stage 2 summary contains duplicate grid-cell/GFED-class records.')
    # A missing Stage 2 row differs from an existing row with zero burned area.
    coverage = links.merge(matched[KEY + ['gfed_id']], on=KEY + ['gfed_id'],
                           how='left', indicator=True, validate='one_to_one')
    absent = coverage.loc[coverage['_merge'] == 'left_only', KEY + ['gfed_id']]
    if not absent.empty:
        raise ValueError(f'Stage 2 summary is missing {len(absent)} required grid-cell/class rows.')
    vals = matched[['gfed_ba_sum_km2', 'grid_area_km2', 'gfed_valid_years']].to_numpy(dtype=float)
    if (not np.isfinite(vals).all() or (matched.gfed_ba_sum_km2 < 0).any() or
            (matched.grid_area_km2 <= 0).any() or
            not matched.gfed_valid_years.between(0, N_YEARS).all() or
            (matched.gfed_valid_years % 1 != 0).any()):
        raise ValueError('Stage 2 summary contains invalid areas or valid-year counts.')
    if (matched.groupby(KEY).grid_area_km2.nunique() != 1).any():
        raise ValueError('Whole grid-cell areas differ between GFED classes for the same candidate.')
else:
    links = pd.DataFrame(columns=KEY + ['gfed_id'])
    matched = pd.DataFrame(columns=[
        'lon', 'lat', 'gfed_id', 'gfed_valid_years',
        'gfed_ba_sum_km2', 'grid_area_km2', 'community_type',
    ])

display(matched.head())


## 6. Cast one vote per grid cell

For each grid, calculate the sum of 21-year burned areas over the compatible classes only. This sum is a denominator for voting; it is never the final fire metric.

`grid vote share = largest compatible class total / sum of compatible class totals`

A vote requires complete records for every compatible class, positive total burned area, a unique largest class, and a share at least MIN_GRID_VOTE_SHARE. A tie abstains even if the share threshold is lowered to 0.50. Zero-fire and incomplete grids also abstain. No annual three-fire-year screen or area-review exclusion is applied during voting; those original filters follow after a primary class is fixed.


In [ ]:
vote_columns = KEY + [
    'compatible_classes_found', 'min_valid_years', 'max_valid_years',
    'compatible_ba_total_km2', 'largest_class_ba_km2',
    'expected_compatible_classes', 'complete_for_vote',
    'grid_vote_share', 'n_largest_classes', 'leading_gfed_id',
    'vote_is_clear', 'voted_gfed_id', 'vote_status',
]

if matched.empty:
    grid_votes = pd.DataFrame(columns=vote_columns)
else:
    grid_votes = matched.groupby(KEY, as_index=False).agg(
        compatible_classes_found=('gfed_id', 'nunique'),
        min_valid_years=('gfed_valid_years', 'min'),
        max_valid_years=('gfed_valid_years', 'max'),
        compatible_ba_total_km2=('gfed_ba_sum_km2', 'sum'),
        largest_class_ba_km2=('gfed_ba_sum_km2', 'max'),
    )
    expected_classes = {}
    for comm, ids in COMMUNITY_TO_GFED.items():
        expected_classes[comm] = len(ids)
    grid_votes['expected_compatible_classes'] = (
        grid_votes['community_type'].map(expected_classes)
    )
    grid_votes['complete_for_vote'] = (
        (grid_votes['compatible_classes_found'] == grid_votes['expected_compatible_classes'])
        & (grid_votes['min_valid_years'] == N_YEARS)
        & (grid_votes['max_valid_years'] == N_YEARS)
    )

    # Identify exact class ties before deciding whether a grid can vote.
    class_maximum = matched.groupby(KEY)['gfed_ba_sum_km2'].transform('max')
    largest_rows = matched.loc[matched['gfed_ba_sum_km2'] == class_maximum]
    leaders = largest_rows.groupby(KEY, as_index=False).agg(
        n_largest_classes=('gfed_id', 'size'),
        leading_gfed_id=('gfed_id', 'first'),
    )
    leaders.loc[leaders['n_largest_classes'] != 1, 'leading_gfed_id'] = np.nan
    grid_votes = grid_votes.merge(leaders, on=KEY, validate='one_to_one')

    grid_votes['grid_vote_share'] = np.nan
    positive_total = grid_votes['compatible_ba_total_km2'] > 0
    grid_votes.loc[positive_total, 'grid_vote_share'] = (
        grid_votes.loc[positive_total, 'largest_class_ba_km2']
        / grid_votes.loc[positive_total, 'compatible_ba_total_km2']
    )
    grid_votes['vote_is_clear'] = (
        grid_votes['complete_for_vote']
        & positive_total
        & (grid_votes['n_largest_classes'] == 1)
        & (grid_votes['grid_vote_share'] >= MIN_GRID_VOTE_SHARE)
    )
    grid_votes['voted_gfed_id'] = np.nan
    clear = grid_votes['vote_is_clear']
    grid_votes.loc[clear, 'voted_gfed_id'] = grid_votes.loc[clear, 'leading_gfed_id']

    grid_votes['vote_status'] = 'below_grid_vote_share'
    grid_votes.loc[grid_votes['n_largest_classes'] > 1, 'vote_status'] = 'tied_largest_ba'
    grid_votes.loc[~positive_total, 'vote_status'] = 'no_positive_compatible_ba'
    grid_votes.loc[~grid_votes['complete_for_vote'], 'vote_status'] = 'incomplete_compatible_gfed'
    grid_votes.loc[clear, 'vote_status'] = 'clear_vote'
    grid_votes = grid_votes.sort_values(KEY)

grid_votes.to_csv(paths['grid_votes'], index=False)
display(grid_votes.head())


## 7. Fix at most one primary GFED class per TEM community

Every clear-voting grid has one equally weighted vote. For a community with multiple compatible classes:

- `winner share = largest class vote count / all clear votes`
- `voter fraction = all clear votes / all LULCC-qualified grids for this community`
- Require MIN_COMMUNITY_VOTES, MIN_COMMUNITY_VOTER_FRACTION, and MIN_COMMUNITY_WINNER_SHARE.
- Equal highest vote counts remain unresolved. There is no class-ID tie-break or fallback to summed fire.

Single-class mappings are retained without requiring voting support because there is no alternative to resolve. They still face all later fire filters. Empty mappings remain disabled. Unresolved multi-class communities are documented and receive no final grid cells.

The primary class is fixed across all locations. This is a burned-area-based mapping decision, not a vegetation-cover validation.


In [ ]:
PRIMARY_GFED = {}
voting_rows = []

for comm in range(1, 24):
    ids = COMMUNITY_TO_GFED[comm]
    n_lulcc = passed[comm]
    community_votes = grid_votes.loc[
        (grid_votes['community_type'] == comm)
        & (grid_votes['vote_is_clear'] == True)
    ]
    n_clear = len(community_votes)
    voter_fraction = 0.0
    if n_lulcc > 0:
        voter_fraction = n_clear / n_lulcc

    vote_counts = community_votes['voted_gfed_id'].value_counts()
    winner_votes = 0
    winner_share = 0.0
    leading_id = None
    tied_winner = False
    if not vote_counts.empty:
        winner_votes = int(vote_counts.max())
        leading_ids = vote_counts.index[vote_counts == winner_votes]
        winner_share = winner_votes / n_clear
        if len(leading_ids) == 1:
            leading_id = int(leading_ids[0])
        else:
            tied_winner = True

    primary_id = None
    if len(ids) == 0:
        decision = 'disabled_mapping'
    elif len(ids) == 1:
        primary_id = ids[0]
        decision = 'single_compatible_class'
    elif n_lulcc == 0:
        decision = 'no_lulcc_candidates'
    elif n_clear == 0:
        decision = 'no_clear_votes'
    elif tied_winner:
        decision = 'tied_community_vote'
    elif n_clear < MIN_COMMUNITY_VOTES:
        decision = 'insufficient_voting_grids'
    elif voter_fraction < MIN_COMMUNITY_VOTER_FRACTION:
        decision = 'insufficient_voter_fraction'
    elif winner_share < MIN_COMMUNITY_WINNER_SHARE:
        decision = 'insufficient_winner_share'
    else:
        primary_id = leading_id
        decision = 'voted_unique_class'

    if primary_id is not None:
        PRIMARY_GFED[comm] = primary_id

    class_vote_text = []
    for gfed_id in ids:
        class_votes = 0
        if gfed_id in vote_counts.index:
            class_votes = int(vote_counts.loc[gfed_id])
        class_vote_text.append(str(gfed_id) + ':' + str(class_votes))
    primary_name = ''
    if primary_id is not None:
        primary_name = GFED_NAMES[primary_id]
    voting_rows.append({
        'community_type': comm,
        'community_name': COMMUNITY_NAMES[comm],
        'n_lulcc_qualified_grids': n_lulcc,
        'n_clear_voting_grids': n_clear,
        'n_abstaining_grids': n_lulcc - n_clear,
        'voter_fraction': voter_fraction,
        'class_vote_counts': '; '.join(class_vote_text),
        'leading_gfed_id': leading_id,
        'winner_votes': winner_votes,
        'winner_share': winner_share,
        'primary_gfed_id': primary_id,
        'primary_gfed_name': primary_name,
        'mapping_decision': decision,
        'min_grid_vote_share': MIN_GRID_VOTE_SHARE,
        'min_community_winner_share': MIN_COMMUNITY_WINNER_SHARE,
        'min_community_voter_fraction': MIN_COMMUNITY_VOTER_FRACTION,
        'min_community_votes': MIN_COMMUNITY_VOTES,
    })

voting_summary = pd.DataFrame(voting_rows)
voting_summary.to_csv(paths['voting'], index=False)
crosswalk = crosswalk.merge(
    voting_summary.drop(columns=['community_name']),
    on='community_type', how='left', validate='one_to_one',
)
crosswalk['selection_enabled'] = crosswalk['primary_gfed_id'].notna()
crosswalk['gfed_class_ids'] = ''
crosswalk['gfed_class_names'] = ''
for comm, primary_id in PRIMARY_GFED.items():
    where = crosswalk['community_type'] == comm
    crosswalk.loc[where, 'gfed_class_ids'] = str(primary_id)
    crosswalk.loc[where, 'gfed_class_names'] = GFED_NAMES[primary_id]

# Keep the original proxy notes; expose unresolved voting decisions as statuses.
unresolved = (
    ~crosswalk['selection_enabled']
    & (crosswalk['mapping_decision'] != 'disabled_mapping')
)
crosswalk.loc[unresolved, 'mapping_status'] = crosswalk.loc[unresolved, 'mapping_decision']
crosswalk.to_csv(paths['crosswalk'], index=False)
display(voting_summary[[
    'community_type', 'community_name', 'n_lulcc_qualified_grids',
    'n_clear_voting_grids', 'voter_fraction', 'winner_share',
    'primary_gfed_id', 'primary_gfed_name', 'mapping_decision',
]])


## 8. Keep only the fixed primary GFED class

Every LULCC-qualified grid in a resolved community uses the same primary class, whether or not that grid voted for it. The other classes are no longer used in its fire metric or fire-year count.

`F = primary-class total burned area / N_YEARS / whole grid-cell area`

No class-average, vegetation-fraction weighting, or cross-class sum is used for F. Unresolved communities are excluded before the original downstream workflow. The compatibility lists remain available in the crosswalk for reference.


In [ ]:
lulcc_eligible = eligible.copy()
eligible = eligible.loc[eligible['community_type'].isin(PRIMARY_GFED)].copy()

primary_rows = []
for comm, primary_id in PRIMARY_GFED.items():
    primary_rows.append({'community_type': comm, 'gfed_id': primary_id})
primary_map = pd.DataFrame(primary_rows, columns=['community_type', 'gfed_id'])

if eligible.empty:
    links = pd.DataFrame(columns=KEY + ['gfed_id'])
    obs = pd.DataFrame()
else:
    links = eligible[KEY].merge(
        primary_map, on='community_type', how='inner', validate='many_to_one',
    )
    primary_matched = matched.merge(
        primary_map, on=['community_type', 'gfed_id'],
        how='inner', validate='many_to_one',
    )
    # Each row now represents exactly one class for one grid/community.
    obs = primary_matched[KEY + ['gfed_valid_years', 'gfed_ba_sum_km2', 'grid_area_km2']].copy()
    obs['gfed_classes_found'] = 1
    obs['gfed_min_valid_years'] = obs['gfed_valid_years']
    obs['gfed_max_valid_years'] = obs['gfed_valid_years']
    obs['gfed_ba_total_km2'] = obs['gfed_ba_sum_km2']
    obs['gfed_ba_total_21y_km2'] = obs['gfed_ba_total_km2']
    obs['gfed_mean_ba_km2'] = obs['gfed_ba_total_km2'] / N_YEARS
    obs['gfed_mean_grid_fraction'] = obs['gfed_mean_ba_km2'] / obs['grid_area_km2']
    obs = obs.drop(columns=['gfed_valid_years', 'gfed_ba_sum_km2'])

# Downstream completeness checks now expect one class, not all alternatives.
ACTIVE_COMMUNITY_TO_GFED = {}
for comm in COMMUNITY_NAMES:
    ACTIVE_COMMUNITY_TO_GFED[comm] = ()
    if comm in PRIMARY_GFED:
        ACTIVE_COMMUNITY_TO_GFED[comm] = (PRIMARY_GFED[comm],)

print('Communities with a fixed primary class:', len(PRIMARY_GFED))
print('LULCC-qualified grids retained for the original fire screen:', len(eligible))


## 9. Count fire years for the primary class

The original chunked SQLite aggregation is retained. Because links now contains only the fixed class, annual areas and distinct positive-fire years refer to that class only. Cross-chunk contributions within the same year are combined. Complete annual totals must reproduce the primary-class summary total.

The annual-area review flag and its optional exclusion setting are unchanged.


In [ ]:
if eligible.empty:
    merged = eligible.merge(
        crosswalk.drop(columns=['community_name']),
        on='community_type', how='left', validate='many_to_one',
    )
    empty_float_columns = [
        'gfed_ba_total_km2', 'gfed_ba_total_21y_km2', 'gfed_mean_ba_km2',
        'gfed_mean_grid_fraction', 'grid_area_km2', 'gfed_max_annual_ba_km2',
        'gfed_annual_check_sum_km2', 'gfed_max_annual_grid_fraction',
        'community_fraction_range',
    ]
    for column in empty_float_columns:
        merged[column] = pd.Series(dtype=float)
    merged['gfed_nonzero_years'] = pd.Series(dtype=int)
    merged['gfed_area_needs_review'] = pd.Series(dtype=bool)
    full = pd.Series(False, index=merged.index, dtype=bool)
    qc_pass = pd.Series(True, index=merged.index, dtype=bool)
else:
    with tempfile.TemporaryDirectory(prefix='stage3_full_1_23_', dir=WORK_DIR) as work:
        conn = sqlite3.connect(Path(work) / 'annual_fire.sqlite')
        try:
            # Limit the SQLite page cache to approximately 16 MB.
            conn.execute('PRAGMA cache_size=-16000')
            conn.execute('''CREATE TABLE fire_year (
                lon REAL,lat REAL,community_type INTEGER,year INTEGER,area REAL,
                PRIMARY KEY (lon,lat,community_type,year)) WITHOUT ROWID''')
            # Insert a new cell/community/year, or add area when that key already exists.
            # This also combines contributions spread across CSV chunks.
            query = '''INSERT INTO fire_year VALUES (?,?,?,?,?)
                ON CONFLICT(lon,lat,community_type,year)
                DO UPDATE SET area=area+excluded.area'''
            need = ['lon', 'lat', 'year', 'gfed_id', 'burned_area_km2']
            for i, chunk in enumerate(pd.read_csv(GFED_ANNUAL, usecols=need,
                                                 chunksize=CSV_CHUNKSIZE), 1):
                if (not chunk.year.between(START_YEAR, END_YEAR).all() or
                        (chunk.year % 1 != 0).any()):
                    raise ValueError('Stage 2 annual data contain invalid years or years outside the configured range.')
                areas = chunk.burned_area_km2.to_numpy(dtype=float)
                if not np.isfinite(areas).all() or (areas <= 0).any():
                    raise ValueError('Stage 2 annual_nonzero must contain only finite, positive burned areas.')
                chunk['lon'] = chunk.lon.round(6)
                chunk['lat'] = chunk.lat.round(6)
                merged = chunk.merge(links, on=['lon', 'lat', 'gfed_id'], how='inner')
                if not merged.empty:
                    group = merged.groupby(KEY + ['year'], as_index=False).burned_area_km2.sum()
                    rows = group[KEY + ['year', 'burned_area_km2']].itertuples(index=False, name=None)
                    with conn:
                        conn.executemany(query, rows)
                if i % 40 == 0:
                    print(f'  GFED annual chunks processed: {i}', flush=True)
            freq = pd.read_sql_query('''SELECT lon,lat,community_type,
                COUNT(*) AS gfed_nonzero_years,
                MAX(area) AS gfed_max_annual_ba_km2,
                SUM(area) AS gfed_annual_check_sum_km2
                FROM fire_year WHERE area>0 GROUP BY lon,lat,community_type''', conn)
        finally:
            conn.close()
    merged = eligible.merge(obs, on=KEY, validate='one_to_one').merge(
        freq, on=KEY, how='left', validate='one_to_one')
    merged['gfed_nonzero_years'] = pd.to_numeric(merged.gfed_nonzero_years).fillna(0).astype(int)
    for col in ('gfed_max_annual_ba_km2', 'gfed_annual_check_sum_km2'):
        merged[col] = pd.to_numeric(merged[col]).fillna(0)
    expected_classes = {}
    for community, classes in ACTIVE_COMMUNITY_TO_GFED.items():
        expected_classes[community] = len(classes)
    merged['expected_gfed_classes'] = merged['community_type'].map(expected_classes)
    full = ((merged.gfed_classes_found == merged.expected_gfed_classes) &
            (merged.gfed_min_valid_years == N_YEARS) &
            (merged.gfed_max_valid_years == N_YEARS))
    # For complete records, annual positive areas must reproduce the multi-year total.
    consistent = np.isclose(merged.gfed_annual_check_sum_km2, merged.gfed_ba_total_km2,
                            rtol=1e-8, atol=1e-7)
    if (full & ~consistent).any():
        raise ValueError('GFED annual and summary areas disagree; check for mixed upstream runs.')
    if (merged.gfed_nonzero_years > N_YEARS).any():
        raise ValueError('The number of fire years exceeds the configured number of years.')
    merged['gfed_max_annual_grid_fraction'] = merged.gfed_max_annual_ba_km2 / merged.grid_area_km2
    merged['gfed_area_needs_review'] = (merged.gfed_max_annual_grid_fraction > ANNUAL_AREA_REVIEW_RATIO)
    merged['community_fraction_range'] = merged.max_community_fraction - merged.min_community_fraction
    merged = merged.merge(crosswalk.drop(columns=['community_name']),
                          on='community_type', how='left', validate='many_to_one')
    qc_pass = ~merged.gfed_area_needs_review
display(merged.head())


## 10. Screen fire activity and calculate community percentiles

Require complete GFED coverage, at least three distinct fire years, and a positive mean burned fraction. Exclude review-flagged candidates only when `EXCLUDE_QC_FLAGGED_CANDIDATES` is `True`.

Within each community, sort by mean burned fraction, longitude, and latitude. Assign empirical plotting positions `(i + 0.5) / n`, where `i` starts at zero. Tied burned fractions can therefore have different positions, matching the original method. These positions define the fire bins; pandas `quantile()` computes the target values separately.


In [ ]:
if eligible.empty:
    final = merged.copy()
else:
    active = ((merged.gfed_nonzero_years >= MIN_FIRE_YEARS) &
              (merged.gfed_mean_grid_fraction > 0))
    use = full & active
    if EXCLUDE_QC_FLAGGED_CANDIDATES:
        use &= qc_pass
    final = merged.loc[use].copy()
# Calculate empirical percentiles separately within each community.
if not final.empty:
    final = final.sort_values(['community_type', 'gfed_mean_grid_fraction', 'lon', 'lat']).copy()
    for comm, group in final.groupby('community_type', sort=True):
        final.loc[group.index, 'empirical_percentile'] = (np.arange(len(group)) + .5) / len(group)
else:
    final['empirical_percentile'] = pd.Series(dtype=float)
final.to_csv(paths['candidates'], index=False)

display(final.head())


## 11. Rank candidates and select low, medium, and high fire cells

Process the groups in this order:

| Group | Target quantile | Empirical-percentile bin |
| --- | --- | --- |
| Low fire | 0.30 | less than 0.40 |
| Medium fire | 0.50 | 0.40 to less than 0.60 |
| High fire | 0.70 | at least 0.60 |

Calculate each target from all eligible candidates for that community. Remove already selected coordinates, then use the requested bin. If the bin has no unused candidates, fall back to all unused candidates and record a flag.

Keep at most 20 candidates closest to the target fire fraction using `nsmallest()`, preserving the source's tie handling. Rank that shortlist by mean community fraction (descending), minimum fraction (descending), fraction range (ascending), distance to target (ascending), longitude, and latitude. These are sequential sorting criteria, not a weighted score.

A cell may be used by different communities, but cannot be reused within one community. A community with fewer than three available cells receives fewer selections.


In [ ]:
rows = []
for info in crosswalk.to_dict('records'):
    comm = info['community_type']
    idx = merged.index[merged.community_type == comm]
    sub = final[final.community_type == comm]
    row = dict(info, input_grid_community_rows=total[comm],
               after_lulcc_screen=passed[comm],
               complete_gfed=int(full.loc[idx].sum()),
               qc_flagged_complete_gfed=int((full.loc[idx] & ~qc_pass.loc[idx]).sum()),
               after_fire_screen=len(sub), n_selected=0)
    if not info['selection_enabled']:
        row['selection_status'] = info['mapping_status']
    elif total[comm] == 0:
        row['selection_status'] = 'no_lulcc_records'
    elif passed[comm] == 0:
        row['selection_status'] = 'failed_lulcc_screen'
    elif not row['complete_gfed']:
        row['selection_status'] = 'failed_gfed_completeness'
    elif sub.empty:
        row['selection_status'] = 'failed_fire_or_optional_qc_screen'
    else:
        row['selection_status'] = 'ready_for_selection'
    rows.append(row)
summary = pd.DataFrame(rows)
winners, rankings = [], []
# Check every community; disabled communities remain documented in the summary.
for comm in range(1, 24):
    sub = final[final.community_type == comm].copy()
    if sub.empty:
        continue
    used = set()
    for label, q in TARGET_QUANTILES.items():
        target = float(sub['gfed_mean_grid_fraction'].quantile(q))

        # Explicit loops replace the row-wise apply(lambda ...) expression.
        available = []
        for lon, lat in zip(sub['lon'], sub['lat']):
            pair = (float(lon), float(lat))
            available.append(pair not in used)
        subset = sub.loc[available].copy()
        if subset.empty:
            continue

        percentile = subset['empirical_percentile']
        if label == 'low_fire':
            pool = subset.loc[percentile < 0.40].copy()
        elif label == 'medium_fire':
            pool = subset.loc[(percentile >= 0.40) & (percentile < 0.60)].copy()
        else:
            pool = subset.loc[percentile >= 0.60].copy()

        # If no unused candidate remains in this bin, use all unused candidates.
        fallback = False
        if pool.empty:
            pool = subset.copy()
            fallback = True

        pool['target_fire_fraction'] = target
        pool['distance_to_target'] = abs(pool['gfed_mean_grid_fraction'] - target)
        pool['distance_to_target_normalized'] = pool['distance_to_target'] / max(target, 1e-12)

        # Keep nsmallest to preserve its original handling of ties at the shortlist boundary.
        # Distance here means fire-fraction difference, not geographic distance.
        shortlist_size = min(SHORTLIST_K, len(pool))
        pool = pool.nsmallest(shortlist_size, 'distance_to_target').copy()
        pool['community_fraction_range'] = (
            pool['max_community_fraction'] - pool['min_community_fraction']
        )

        # Compare fields in order; these criteria are not combined into a weighted score.
        ranking_columns = [
            'mean_community_fraction', 'min_community_fraction',
            'community_fraction_range', 'distance_to_target', 'lon', 'lat',
        ]
        ranking_ascending = [False, False, True, True, True, True]
        pool = pool.sort_values(ranking_columns, ascending=ranking_ascending).copy()
        pool['fire_group'] = label
        pool['target_quantile'] = q
        pool['shortlist_rank'] = np.arange(1, len(pool) + 1)
        pool['fallback_no_quantile_bin'] = fallback
        best = pool.head(1).copy()
        rankings.append(pool)
        best['selection_method'] = 'automatic'
        best['selection_reason'] = 'GFED percentile band + community dominance/stability'
        best['manual_override_outside_quantile_bin'] = False
        winners.append(best)
        used.add((float(best.iloc[0].lon), float(best.iloc[0].lat)))
# Write headers even when empty; do not invent selections for missing fire groups.
extra = ['target_fire_fraction', 'distance_to_target', 'distance_to_target_normalized',
         'fire_group', 'target_quantile', 'shortlist_rank', 'fallback_no_quantile_bin']
if rankings:
    shortlist = pd.concat(rankings, ignore_index=True)
else:
    shortlist = pd.DataFrame(columns=list(final.columns) + extra)

if winners:
    selected = pd.concat(winners, ignore_index=True)
else:
    selected = pd.DataFrame(columns=list(shortlist.columns) + [
        'selection_method', 'selection_reason', 'manual_override_outside_quantile_bin',
    ])
shortlist.to_csv(paths['shortlist'], index=False)
display(selected)


## 12. Apply optional manual replacements and save results

The first run creates an empty `03_manual_overrides.csv` in `RESULT_DIR`. Enter community_type, fire_group, lon, lat, and a scientific reason to replace an automatic selection. Restart the kernel and run all cells again.

Replacements must be eligible and refer to an existing selected group. The final result cannot reuse a cell within a community. Manual replacements may lie outside the original bin and receive an explicit flag. Automatic shortlist results remain unchanged.

All replacement candidates use the community's already fixed primary class. Manual replacements cannot change the GFED mapping.


In [ ]:
template = ['community_type', 'fire_group', 'lon', 'lat', 'reason']
if not paths['overrides'].exists():
    pd.DataFrame(columns=template).to_csv(paths['overrides'], index=False)
else:
    override = pd.read_csv(paths['overrides'])
    if not set(template).issubset(override.columns):
        raise ValueError(f'The manual override table requires columns: {template}')
    if override.duplicated(['community_type', 'fire_group']).any():
        raise ValueError('Only one override is allowed per community/fire_group.')
    out = selected.copy()
    out['manual_override_outside_quantile_bin'] = False
    for row in override.itertuples(index=False):
        if pd.isna(row.community_type) or float(row.community_type) % 1:
            raise ValueError('Manual override community_type must be an integer.')
        comm, label = int(row.community_type), str(row.fire_group)
        if label not in TARGET_QUANTILES or not isinstance(row.reason, str) or not row.reason.strip():
            raise ValueError('An override requires a valid fire_group and a nonempty scientific reason.')
        old = out[(out.community_type == comm) & (out.fire_group == label)]
        replacement = final[(final.community_type == comm) &
                                  (final.lon == round(float(row.lon), 6)) &
                                  (final.lat == round(float(row.lat), 6))]
        if len(old) != 1 or len(replacement) != 1:
            raise ValueError(f'The replacement is ineligible or the original selection does not exist: {comm}, {label}, {row.lon}, {row.lat}')
        rep = replacement.iloc[0].copy()
        for col in ('fire_group', 'target_quantile', 'target_fire_fraction'):
            rep[col] = old.iloc[0][col]
        rep['distance_to_target'] = abs(rep.gfed_mean_grid_fraction - rep.target_fire_fraction)
        rep['distance_to_target_normalized'] = rep.distance_to_target / max(rep.target_fire_fraction, 1e-12)
        # A manual replacement may be outside the shortlist, so it has no automatic rank.
        rep['shortlist_rank'] = np.nan
        rep['fallback_no_quantile_bin'] = old.iloc[0]['fallback_no_quantile_bin']
        p = rep.empirical_percentile
        if label == 'low_fire':
            in_bin = p < 0.40
        elif label == 'medium_fire':
            in_bin = 0.40 <= p < 0.60
        else:
            in_bin = p >= 0.60
        rep['manual_override_outside_quantile_bin'] = not in_bin
        rep['selection_method'] = 'manual_override'
        rep['selection_reason'] = row.reason.strip()
        for col in out.columns:
            if col in rep.index:
                out.loc[old.index[0], col] = rep[col]
    if out.duplicated(KEY).any():
        raise ValueError('Manual overrides reuse a grid cell within the same community.')
    selected = out
selected = selected.sort_values(['community_type', 'target_quantile'])
selected.to_csv(paths['selected'], index=False)
for comm, n in selected.community_type.value_counts().items():
    where = summary.community_type == comm
    summary.loc[where, 'n_selected'] = int(n)
    if n == 3:
        summary.loc[where, 'selection_status'] = 'selected_3_groups'
    else:
        summary.loc[where, 'selection_status'] = 'selected_fewer_than_3'
summary.to_csv(paths['summary'], index=False)
print(summary[['community_type', 'community_name', 'after_lulcc_screen',
               'after_fire_screen', 'n_selected', 'selection_status']].to_string(index=False))
print('Stage 3 output directory: ', RESULT_DIR, flush=True)
display(selected)
display(summary)


## Outputs and interpretation

| File | Contents |
| --- | --- |
| `03_gfed_grid_votes.csv` | One row per mapped LULCC-qualified grid/community, compatible-area totals, vote share, vote or abstention reason |
| `03_community_gfed_voting.csv` | All 23 communities, class vote counts, support fractions, thresholds, primary class, and decision |
| `03_community_gfed_crosswalk.csv` | Original compatibility lists and proxy notes, plus the fixed primary class or unresolved status |
| `03_all_eligible_candidates.csv` | Original fire-screen results with percentiles, primary class, and area-review flags |
| `03_ranked_shortlists.csv` | Original automatic shortlists and ranks |
| `03_selected_observation_only.csv` | Final low/medium/high selections after optional manual replacements |
| `03_community_screening_summary.csv` | Original screening counts and selection statuses plus voting information |
| `03_manual_overrides.csv` | Preserved replacements or an empty template |

The denominator of voter_fraction includes all LULCC-qualified grids for that community, including zero-fire and incomplete grids. winner_share uses only clear votes. Threshold comparisons are inclusive. Multi-class communities with ties or insufficient support have no primary class and no selections. Single-class communities bypass voting thresholds, but not the original fire screens.

The selected GFED class is fixed at community level and is not changed to rescue a failed fire screen. Fire levels remain relative to the final eligible candidates within each TEM community. At most three cells are selected per resolved community, not three per compatible GFED class. Voting is based on the same observation period and is a selection rule, not independent ecological validation. It can discard secondary-class fire and does not create community-specific burned-area observations.

After changing thresholds or manual replacements, restart the kernel and run all cells. When the primary mapping changes, review existing manual replacements: they must still be eligible under the new mapping.


## Verification scope

The code contains no user-defined functions, lambdas, or classes. Code cells and the companion script were syntax-checked. Synthetic tests cover class voting and abstention, community support thresholds, ties, zero and incomplete records, one-class and disabled mappings, primary-class-only fire years, empty results, manual replacements, and preserved downstream ranking against the previous notebook with the same fixed mapping. Detailed results are in the accompanying usage note.

No real server Stage 1 or Stage 2 CSV inputs have been executed for this notebook.
